# Welcome to NER Finetuning with HF Transformers 

First of all, we need to import all necessary libraries and modules.

## Contents 
1. Configurations (Resources, Training, Optimization, Evaluation, Saving results)
2. Prepare training & dataset
3. Train (Finetune) a NER Model
4. Optimization
5. Evaluation

In [1]:
# import modules
import config
import train
import eval_opt

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configurations

In this section, we set all of the configurations for the following steps (training, optimization, evaluation). Depending on which parts you want to run and which to leave out, corresponding config parts can be skipped if not needed.

### Resources

We need to state which resources we want to use for NER finetuning, namely the `dataset` and `model`. Therefore, we use the `Resource` class and define the `path` (either to a HF or a local directory) and `source` (`hf` or `local`) of our resources. By applying `set_name()`, the model or dataset name gets set automatically, otherwise the `model.name` can also be updated manually.

In [2]:
"""
#uncomment and delete other section later for explaination purposes
model = config.Resource(path="test/test", source="hf")
model.name = "my_model_name"

dataset = config.Resource(path="dir/dataset", source="local")
dataset.set_name()
"""

model = config.Resource(path="prajjwal1/bert-tiny", source="hf")
model.set_name()

dataset = config.Resource(path="conll2003", source="hf")
dataset.set_name()

In [3]:
#check our model and dataset configurations again
print(model.info())
print(dataset.info())

bert-tiny will be loaded from prajjwal1/bert-tiny (via hf).
conll2003 will be loaded from conll2003 (via hf).


### Training

You can define a variable for the training parameters based on our default `TrainingParams` config and try to improve results using evaluation/optimization techniques, or start by overwriting the parameters for training manually. Training will only be performed once on the training dataset and evaluated once on the validation dataset. For more sophisticated methods, you can additionally run the Optimization and Evaluation parts to find ideal hyperparameter combinations and perform a more reliable evaluation.

In [4]:
training_params = config.TrainingParams()
training_params.batch_size = 32
training_params.__dict__

{'batch_size': 32,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'weight_decay': 0.01}

### Optimization

For optimization (hyperparameter search) we use the HF Trainer API (https://huggingface.co/docs/transformers/hpo_train#hyperparameter-search-using-trainer-api) and optuna (https://optuna.readthedocs.io/en/stable/index.html, `pip install optuna`). We can access all of the search parameters, as they are stored in a dict structure, and also adapt the number of trials `n_trials` to run. 

In [5]:
optimize_params = config.OptimizeParams()

optimize_params.hp_space["learning_rate"]["param_type"]
optimize_params.__dict__

{'hp_space': {'learning_rate': {'param_type': 'float',
   'borders': [1e-06, 0.0001]},
  'per_device_train_batch_size': {'param_type': 'categorical',
   'borders': [16, 32, 64, 128]}},
 'n_trials': 20}

In [12]:
#to remove an hyperparameter from the hp_space dict, use pop() or similar
optimize_params.hp_space.pop("learning_rate")
optimize_params.__dict__

{'hp_space': {'per_device_train_batch_size': {'param_type': 'categorical',
   'borders': [16, 32, 64, 128]}},
 'n_trials': 20}

### Evaluation and optimization

* Evaluation: Stratified k-fold Cross-Validation - one of ["None", k] (https://discuss.huggingface.co/t/k-fold-cross-validation/5765/5)
* Optimization:
  * https://huggingface.co/docs/transformers/hpo_train
  * https://medium.com/carbon-consulting/transformer-models-hyperparameter-optimization-with-the-optuna-299e185044a8
  * Grid Search (GridSearchCV?), Line Search, hyperopt
* Analyze grid search results using Heatmap or similar

### Saving results
* save configuration
* create/save plots

## 2. Prepare training & dataset

In [6]:
train.set_torch_device()

device(type='cuda')

In [7]:
ner_dataset = train.load_ner_dataset(dataset.path, dataset.source)
ner_dataset["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

In [8]:
tokenizer = train.get_tokenizer(model.path)

In [9]:
tokenized_dataset = train.prepare_dataset(ner_dataset, tokenizer)
label_list = train.get_label_list(ner_dataset)

In [10]:
label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

## 3. Train (Finetune) a NER Model

To finetune our model for NER, we first need to define our `ner` task (see train.py). 

In [12]:
trained_ner_model = train.train_model(model.path, label_list, training_params, tokenized_dataset, tokenizer)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.908900,0.532401,0.370214,0.343998,0.356625,0.866173
2,0.540200,0.454397,0.437153,0.431480,0.434298,0.881583
3,0.457900,0.434780,0.456689,0.457098,0.456894,0.886476


/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


## 4. Optimization

In [11]:
eval_opt.optimize(optimize_params, training_params, model.path, label_list, tokenizer, tokenized_dataset)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[I 2025-03-19 16:52:22,949] A new study created in memory with name: no-name-faf8cd39-9dad-40b2-bb0d-d1fe87450788
Some weights of BertForTokenClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.568104,0.357731,0.316031,0.335590,0.860565
2,0.909200,0.468915,0.404470,0.390760,0.397496,0.876340
3,0.521100,0.446456,0.431357,0.420741,0.425983,0.882743


/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
[I 2025-03-19 16:53:02,520] Trial 0 finished with value: 2.1608225642104992 and parameters: {'learning_rate': 2.3694963513295655e-05, 'per_device_train_batch_size': 32}. Best is trial 0 with value: 2.1608225642104992.
Some weights of BertForTokenClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,1.702631,0.076098,0.017452,0.028392,0.776701


[W 2025-03-19 16:53:15,264] Trial 1 failed with parameters: {'learning_rate': 2.0993064867573863e-06, 'per_device_train_batch_size': 32} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/accelerate/utils/operations.py", line 156, in send_to_device
    return tensor.to(device, non_blocking=non_blocking)
TypeError: BatchEncoding.to() got an unexpected keyword argument 'non_blocking'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/transformers/integrations/integration_utils.py", line 247, in _objective
    trainer.train

KeyboardInterrupt: 

## 5. Evaluation